<a href="https://colab.research.google.com/github/zami456/expLB/blob/claude%2Fcolab-activation-download-drive/notebooks/colab_extract_activations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download Liars' Bench probe activations on Colab → save to Google Drive

This notebook **drives** the white-box Liars' Bench probe evaluation
(`deception_detection/scripts/experiment.py`) from Colab, running the benchmark's **base-model**
scenario config unchanged, while **all code stays in GitHub** and **all outputs go to Google Drive**.

```
VS Code (edit) --git push--> GitHub <--git pull-- Colab (run) --> Google Drive (outputs)
```

**How the workflow fits together**
- Edit code in **VS Code** on your laptop, then `git push`.
- In Colab, re-run the *Clone / update repo* cell (`git pull`) to get your changes, then re-run.
- The model runs **remotely on NDIF** (`model.trace(..., remote=True)`), so Colab only orchestrates
  and downloads the activations — you do **not** need a big GPU here.
- Activations + results are written to **Drive**, and runs are **resumable**: activations are cached
  **per batch** with atomic writes, so on a Colab disconnect you reconnect and re-run — completed
  batches/scenarios load from the Drive cache and only the remainder is re-downloaded from NDIF.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.** (Even though compute is remote, a CUDA runtime avoids
   an error in the entry script's memory-logging line. A free T4 is plenty.)
2. Add your keys under **🔑 Secrets** (left sidebar): `NDIF_API_KEY` (from https://login.ndif.us) and
   `HF_TOKEN`. Toggle *Notebook access* on for each.
3. **Base models only.** With base weights you can run the one Liars' Bench scenario that uses a base
   model — **`XXXX-9`** (Llama-70B-3.3: harm-refusal + instructed-dishonesty). The other scenarios
   (`XXXX-15/16` convincing-game & gender-secret, `XXXX-17/18/19` sleeper-agents) need **fine-tuned /
   LoRA model-organism adapters** you don't have, so they're disabled in the config cell.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuration

Edit these to match your repo, branch, models and config file.

In [2]:
# --- Repo -------------------------------------------------------------------
REPO_URL = 'https://github.com/zami456/expLB.git'
BRANCH   = 'claude/colab-activation-download-drive'  # branch you develop on in VS Code
REPO_DIR = '/content/expLB'
# All extraction commands run from the vendored probes package:
WORKDIR  = REPO_DIR + '/src/probes/deception-detection'

# --- Output locations -------------------------------------------------------
DRIVE_BASE     = '/content/drive/MyDrive/expLB'
OUT            = DRIVE_BASE + '/results'          # RESULTS on Drive (detector.pt, scores, plots/) -- small, kept
# Activation cache on Colab-LOCAL scratch, NOT Drive: the per-batch hidden states of a 70B model are
# many GB and would fill Drive. Local /content has ~100+ GB. Trade-off: the cache is wiped when the
# runtime ends, so cross-session resume is lost -- but RESULTS still resume from Drive. To keep the
# cache on Drive instead (persists, but large), set: ACTS_CACHE_DIR = DRIVE_BASE + '/activations_cache'
ACTS_CACHE_DIR = '/content/activations_cache'

# --- What to run: the Liars' Bench scenario configs -------------------------
# Each XXXX-* config IS a Liars' Bench scenario, run UNCHANGED -- we only redirect outputs.
# BASE-MODEL ONLY: XXXX-9 (base Llama-70B-3.3) is the only scenario runnable with base weights.
CFG_DIR = 'deception_detection/scripts/configs'
CONFIGS = [
    f'{CFG_DIR}/XXXX-9.yaml',    # harm-refusal + instructed-dishonesty | Llama-70B-3.3 base -> runs on NDIF
    # f'{CFG_DIR}/XXXX-15.yaml',  # needs LoRA adapter (not base) -> local only
    # f'{CFG_DIR}/XXXX-16.yaml',  # needs LoRA adapter (not base) -> local only
    # f'{CFG_DIR}/XXXX-17.yaml',  # needs Gemma LoRA adapter (not base) -> local only
    # f'{CFG_DIR}/XXXX-18.yaml',  # needs Gemma LoRA adapter (not base) -> local only
    # f'{CFG_DIR}/XXXX-19.yaml',  # needs Gemma LoRA adapter (not base) -> local only
]

# --- Which base model to extract with --------------------------------------
# None -> each config's own model (XXXX-9 = llama-70b-3.3). To do the other base models afterwards,
# set this and re-run cell 6; each writes its own <config>__<model> folder. NDIF-hosted base models:
# 'llama-70b-3.3', 'llama-70b-3.1', 'qwen-72b'  (llama-70b-3.1 is usually the most reliably deployed)
MODEL_OVERRIDE = 'llama-70b-3.1'

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(ACTS_CACHE_DIR, exist_ok=True)
print('Results (Drive) ->', OUT)
print('Acts cache (local scratch) ->', ACTS_CACHE_DIR)
print(f"{len(CONFIGS)} config(s) queued; model = {MODEL_OVERRIDE or 'config default (llama-70b-3.3)'}")

Results (Drive) -> /content/drive/MyDrive/expLB/results
Acts cache (local scratch) -> /content/activations_cache
1 config(s) queued; model = llama-70b-3.1


## 3. Clone / update the repo

Run this cell every time you push new code from VS Code — it clones on the first run and `git pull`s
afterwards.

**If `zami456/expLB` is private** (a `git clone` exit-128 / "Authentication failed" / "not found" means
it is), add a **`GH_TOKEN`** Colab Secret: a GitHub Personal Access Token with read access to the repo
(fine-grained → *Contents: Read*, or classic → `repo` scope). The cell uses it automatically and never
prints it. Public repos need no token.

In [3]:
import os, subprocess
from google.colab import userdata

# PRIVATE repo -> needs a GitHub token. Add a Colab Secret named exactly GH_TOKEN and toggle
# "Notebook access" ON. (GitHub PAT: fine-grained with "Contents: Read" on zami456/expLB, or
# a classic token with the 'repo' scope.)
GH_TOKEN = None
try:
    GH_TOKEN = (userdata.get('GH_TOKEN') or '').strip()
except Exception as e:
    print(f"Could not read GH_TOKEN secret -> {type(e).__name__}: {e}")

if not GH_TOKEN:
    raise SystemExit(
        "GH_TOKEN not available. In the Colab key (Secrets) panel: add a secret named exactly "
        "GH_TOKEN holding your GitHub token, toggle 'Notebook access' ON, then re-run this cell. "
        "(Only skip this if the repo is public.)"
    )
print(f"GH_TOKEN loaded (length {len(GH_TOKEN)}).")

auth_url = REPO_URL.replace('https://', f'https://x-access-token:{GH_TOKEN}@')

def git(*cmd):
    r = subprocess.run(['git', *cmd], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(GH_TOKEN, '***')  # scrub token from error text
        raise RuntimeError(f"git {' '.join(cmd[:2])} failed (exit {r.returncode}):\n{msg}")
    return r.stdout

if not os.path.isdir(REPO_DIR + '/.git'):
    git('clone', '--branch', BRANCH, auth_url, REPO_DIR)
else:
    git('-C', REPO_DIR, 'remote', 'set-url', 'origin', auth_url)
    git('-C', REPO_DIR, 'fetch', 'origin', BRANCH)
    git('-C', REPO_DIR, 'checkout', BRANCH)
    git('-C', REPO_DIR, 'pull', 'origin', BRANCH)

print('On:', git('-C', REPO_DIR, 'log', '--oneline', '-1').strip())

GH_TOKEN loaded (length 40).
On: 0d219b0 Cache to local scratch + per-dataset example progress bar


## 4. Install dependencies

Installs the `deception_detection` package (pulls in `nnsight`), then restores **numpy 2.x** — the
package pins numpy 1.26 but Colab's torch/transformers need numpy 2.x, and the mismatch otherwise
crashes `import nnsight` with a `numpy.dtype size changed` ABI error.

**After this cell finishes, do `Runtime → Restart session`**, then run cells **1, 2, 5, 6** (skip 3
and 4 — the repo and packages are already on disk). You only need to re-run this cell when
dependencies change.

In [4]:
!pip install -q -e {WORKDIR}
# The probes package pins numpy~=1.26, but Colab's torch/transformers are built against numpy 2.x.
# Letting numpy stay at 1.x causes a "numpy.dtype size changed" ABI crash on `import nnsight`.
# Restore numpy 2.x, then RESTART the runtime so the fresh kernel loads it.
!pip install -q "numpy>=2.1,<3"
print("\nInstall done. IMPORTANT: Runtime -> Restart session, then run cells 1, 2, 5, 6 "
      "(skip 3 and 4 -- repo + packages are already on disk).")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Re-install numpy 2.1

To ensure `numpy` version `2.1` is correctly installed and loaded, run the cell below. **Crucially, after this cell completes, you MUST go to `Runtime -> Restart session`** for the changes to take effect. Afterwards, run cells 1, 2, 5, and 6.

In [7]:
!pip install -q "numpy==2.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 99.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deception-detection 0.0.1 requires numpy~=1.26.4, but you have numpy 2.1.0 which is incompatible.
tobler 0.14.0 requires tqdm>=4.67, but you have tqdm 4.66.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.0 which is incompatible.


## 5. Secrets & remote (NDIF) configuration

Reads `NDIF_API_KEY` / `HF_TOKEN` from Colab Secrets, sets them in the environment, points the
activation cache at Drive (`ACTS_CACHE_DIR`), and registers the NDIF key with nnsight (persisted to
nnsight's config so the subprocess runs in the next cell are authenticated).

In [5]:
import os
from google.colab import userdata

os.environ['NDIF_API_KEY']   = userdata.get('NDIF_API_KEY')
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['ACTS_CACHE_DIR'] = ACTS_CACHE_DIR  # read by Activations.from_model (get_acts_cache_dir)

import nnsight
nnsight.CONFIG.set_default_api_key(os.environ['NDIF_API_KEY'])
print('NDIF key + HF token configured; cache ->', os.environ['ACTS_CACHE_DIR'])

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [6]:
import numpy as np
print(f"Numpy version: {np.__version__}")

Numpy version: 2.0.2


## 6. Run — extract activations, train the probe, score, and plot

Each `run` does the **full pipeline** in one go and writes it all to Drive under `results/<name>/`:
- downloads the activations (cached per batch),
- **trains** the linear probe → `detector.pt`,
- computes eval + control **scores** → `scores.json`, `control_scores.json`,
- generates **figures** → `plots/` (ROC curves, AUROC, violin, per-dataset plots).

By default only base-model `XXXX-9` is enabled and it runs on **Llama-70B-3.3**. **tqdm download bars
stream into this cell.** To do the other base models afterwards, set `MODEL_OVERRIDE` in cell 2 to
`'llama-70b-3.1'` (then `'qwen-72b'`) and re-run this cell — each writes its own `XXXX-9__<model>` folder.

**Resume:** reconnect after a disconnect and re-run — completed per-batch `.pt` files load from
`ACTS_CACHE_DIR` (no NDIF re-download), and a finished `detector.pt` / `scores.json` in the
stable-named results folder is skipped.

In [ ]:
import subprocess, os, sys, time
from pathlib import Path

MAX_RETRIES = 3          # NDIF "Model deployment evicted" is transient -> retry
RETRY_WAIT  = 90         # seconds between attempts (gives NDIF time to redeploy)

def run_stream(cmd, cwd):
    """Run a subprocess, forwarding its combined stdout+stderr to the cell in real time.
    Reads raw chunks (not lines) so tqdm's carriage-return progress bars animate live in Colab."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    fd = p.stdout.fileno()
    while True:
        chunk = os.read(fd, 4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode('utf-8', errors='replace'))
        sys.stdout.flush()
    return p.wait()

results = {}
for cfg in CONFIGS:
    stem = Path(cfg).stem
    if MODEL_OVERRIDE:
        name = f"{stem}__{MODEL_OVERRIDE}"      # model-specific folder so models don't overwrite
        model_args = [f'--model_name={MODEL_OVERRIDE}']
    else:
        name = stem                              # canonical run (config's own model)
        model_args = []
    cmd = [
        'python', '-u', 'deception_detection/scripts/experiment.py', 'run',
        '--config-file', cfg, *model_args, f'--name={name}', f'--folder={OUT}',
    ]
    rc = 1
    for attempt in range(1, MAX_RETRIES + 1):
        print(f"\n{'='*72}\n=== {name}  (attempt {attempt}/{MAX_RETRIES})\n{'='*72}", flush=True)
        rc = run_stream(cmd, WORKDIR)
        if rc == 0:
            break
        if attempt < MAX_RETRIES:
            print(f"\n[exit {rc}] retrying in {RETRY_WAIT}s — cached batches are kept, so it resumes "
                  "from where it stopped...", flush=True)
            time.sleep(RETRY_WAIT)
    results[name] = 'OK' if rc == 0 else f'FAILED (exit {rc})'

print(f"\n{'='*72}\nSummary — results, scores and plots/ are on Drive at {OUT}\n{'='*72}")
for name, status in results.items():
    print(f"  {name:<28} {status}")

## 7. (Optional) Aggregate across runs — dashboard, combined CSV, LaTeX table

Each `run` already writes its own scores + `plots/`. This cell **combines every run** under your Drive
`results/` folder (all scenarios and models you've completed so far) into:
- a **multi-experiment dashboard** `multi_experiment_dashboard_<ts>.html`,
- a **combined CSV** `multi_experiment_results_<ts>.csv`,
- a best-effort **LaTeX table** `white_box_results.tex` (AUROC / balanced-acc / recall@1%FPR).

Run it after cell 6, and re-run whenever you add another model (e.g. after switching `MODEL_OVERRIDE`)
— it re-scans the folder. All outputs land next to your results on Drive. The LaTeX table is tuned to
the *full* paper matrix, so with base-model-only runs it will show empty (—) cells; the dashboard +
combined CSV are the robust aggregate view for a partial run.

In [ ]:
# Aggregate every run under the Drive results folder into one dashboard + combined CSV,
# and (best-effort) a LaTeX AUROC/accuracy table. Run AFTER cell 6 has produced one or more
# results/<name>/ folders. Safe to re-run -- it re-scans OUT (e.g. after adding another model).
import subprocess, glob, shutil
from pathlib import Path

# 1) Multi-experiment HTML dashboard + combined CSV over ALL runs under OUT.
subprocess.run(
    ['python', 'deception_detection/scripts/generate_html_dashboard.py',
     '--multi', '--search_dirs', OUT, '--no_zip'],
    cwd=WORKDIR, check=False,
)

dash = sorted(glob.glob(str(Path(OUT) / 'multi_experiment_dashboard_*.html')))
csvs = sorted(glob.glob(str(Path(OUT) / 'multi_experiment_results_*.csv')))
print('\nDashboard   :', dash[-1] if dash else '(none)')
print('Combined CSV:', csvs[-1] if csvs else '(none)')

# 2) Best-effort LaTeX table (white_box_results.tex). make_white_box_tables.py is tuned to the FULL
#    benchmark matrix, so with base-model-only / partial runs the table will have empty (---) cells.
if csvs:
    shutil.copy(csvs[-1], Path(WORKDIR) / 'multi_experiment_dashboard.csv')  # name it expects
    rc = subprocess.run(['python', 'deception_detection/scripts/make_white_box_tables.py'],
                        cwd=WORKDIR, check=False).returncode
    tex = Path(WORKDIR) / 'white_box_results.tex'
    if rc == 0 and tex.exists():
        shutil.copy(tex, Path(OUT) / 'white_box_results.tex')
        print('LaTeX table :', Path(OUT) / 'white_box_results.tex')
    else:
        print('LaTeX table : skipped (needs the full benchmark matrix)')
else:
    print('No runs found under OUT yet -- run cell 6 first.')

## Notes

- **Watch progress**: tqdm bars above + browse the Drive folders (`results/`, `activations_cache/`).
- **Change code**: edit in VS Code → `git push` → re-run cell **3** (`git pull`) → re-run cell **6**.
- **The configs are the benchmark**: `XXXX-*.yaml` are the Liars' Bench scenarios, run unchanged. To
  add/remove scenarios, edit the `CONFIGS` list — not the datasets inside a config.
- **Resume granularity**: the cache unit is a *batch* (`BATCH_SIZE_ACTS`, default 32); a mid-batch
  interruption re-downloads only that batch (atomic writes prevent corrupt partial caches).
- **Colab idle**: sessions disconnect on idle (~90 min) or at runtime caps — expected; the resume
  design makes it harmless. Don't use anti-idle scripts (against Colab ToS).